## Deep Convolutional Generative Adversarial Networks

In [ ]:
import os
import statistics

import torch
import torchvision
from torchvision.transforms.functional import to_pil_image
from cleanfid import fid
from IPython.display import display

from models import Generator
from utils.gen_utils import gen

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
def load_checkpoint(checkpoint_path, device="cuda"):
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model = Generator(
		img_size=checkpoint["img_size"],
		in_channels=checkpoint["generator_in_channels"],
		latent_dim=checkpoint["latent_dim"],
	)
    model = model.to(device)
    model.load_state_dict(checkpoint["G"])
    return model, checkpoint["latent_dim"]

### AFHQ-Cat-64x64

In [ ]:
afhq_cat_checkpoint_path = os.path.join("checkpoints", "afhq-cat_500.pth")

In [ ]:
afhq_cat_model, afhq_cat_latent_dim = load_checkpoint(afhq_cat_checkpoint_path, device=device)

In [ ]:
afhq_cat_fid_scores = []
for _ in range(10):
	afhq_cat_fid_score = fid.compute_fid(
		gen=lambda z: gen(afhq_cat_model, z, device),
		dataset_name="afhq-cat-64x64",
		dataset_res=64,
		num_gen=50_000,
		dataset_split="custom"
  	)
	afhq_cat_fid_scores.append(afhq_cat_fid_score)

In [ ]:
afhq_cat_fid_mean = statistics.mean(afhq_cat_fid_scores)
print(f"Mean: {afhq_cat_fid_mean}")

In [ ]:
afhq_cat_fid_stdev = statistics.stdev(afhq_cat_fid_scores)
print(f"Standard Deviation: {afhq_cat_fid_stdev}")

In [ ]:
noise = torch.randn(9*9, afhq_cat_latent_dim, 1, 1, device=device)

with torch.no_grad():
    fake_images = afhq_cat_model(noise)

grid = torchvision.utils.make_grid(
    fake_images.detach().cpu(),
    nrow=9,
    normalize=True,
)

pil_image = to_pil_image(grid)
display(pil_image)